# TimelineQA Experiments Colab

Run cells one by one. Start with the smoke test before running a full model evaluation.

In [6]:
# Cell 1: Check GPU
!nvidia-smi

Tue Jun 30 17:48:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Cell 3: Clone or pull repo
import os
import subprocess

repo_dir = '/content/timelineqa_experiments'
repo_url = 'https://github.com/KYPKRISHNAREDDY/timelineqa_experiments.git'

if os.path.exists(repo_dir):
    subprocess.run(['git', '-C', repo_dir, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, repo_dir], check=True)

os.chdir(repo_dir)
print('Current folder:', os.getcwd())

Current folder: /content/timelineqa_experiments


In [9]:
# Cell 4: Install requirements
!pip install -r requirements.txt

In [10]:
# Cell 5: Read HF_TOKEN from Colab Secrets and set cache paths
from google.colab import userdata
import os
import subprocess

hf_token = userdata.get('HF_TOKEN')
os.environ['HF_HOME'] = '/content/drive/MyDrive/timelineqa_hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/content/drive/MyDrive/timelineqa_hf_cache'

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    subprocess.run(['hf', 'auth', 'login', '--token', hf_token], check=True)
else:
    print('HF_TOKEN not found. Public models may still work, but gated models will fail.')

In [11]:
# Cell 6: Clone official TimelineQA
!bash scripts/00_clone_timelineqa.sh

Cloning into 'original/TimelineQA'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 62 (delta 22), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 63.14 KiB | 10.52 MiB/s, done.
Resolving deltas: 100% (22/22), done.
Cloned official TimelineQA repo into original/TimelineQA.


In [12]:
# Cell 7: Make n=50 atomic sample
!python scripts/02_make_samples.py --task atomic --n 50 --seed 42

Wrote 50 atomic samples to /content/timelineqa_experiments/data/samples/atomic_n50.jsonl


In [13]:
# Cell 8: Smoke test Qwen 0.5B with --limit 3
!python scripts/04_run_model.py \
  --task atomic \
  --sample data/samples/atomic_n50.jsonl \
  --model_id Qwen/Qwen2.5-0.5B-Instruct \
  --backend hf \
  --retriever bm25 \
  --top_k 5 \
  --max_new_tokens 32 \
  --temperature 0 \
  --output outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --limit 3

Loading one model: Qwen/Qwen2.5-0.5B-Instruct
config.json: 100% 659/659 [00:00<00:00, 3.07MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 7.58MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 12.1MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 12.8MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 19.5MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:11<00:00, 87.6MB/s]
Loading weights: 100% 290/290 [00:02<00:00, 116.43it/s]
generation_config.json: 100% 242/242 [00:00<00:00, 1.25MB/s]
Running model: 100% 3/3 [00:06<00:00,  2.18s/it]
Saved predictions to outputs/predictions/atomic_n50_qwen05b_bm25.jsonl


In [14]:
# Cell 9: Evaluate
!python scripts/05_evaluate.py \
  --predictions outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --task atomic \
  --output outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 3,
  "exact_match": 0.0,
  "token_f1": 0.0303030303030303,
  "denotation_accuracy": 0.0,
  "recall_at_1": 0.3333333333333333,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 2.1825031016666876,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T17:49:23.761907+00:00"
}
Saved metrics to outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json


In [15]:
# Cell 10: Copy outputs to Drive
!mkdir -p /content/drive/MyDrive/timelineqa_results
!cp -r outputs /content/drive/MyDrive/timelineqa_results/

In [16]:
# Cell 11: Show metrics
!cat outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 3,
  "exact_match": 0.0,
  "token_f1": 0.0303030303030303,
  "denotation_accuracy": 0.0,
  "recall_at_1": 0.3333333333333333,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 2.1825031016666876,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T17:49:23.761907+00:00"
}

In [17]:
import json

path = "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        print("="*80)
        print("Q:", row["question"])
        print("GOLD:", row["gold_answer"])
        print("PRED:", row["predicted_answer"])
        print("RETRIEVED IDS:", row["retrieved_episode_ids"])
        print("GOLD EVIDENCE:", row.get("evidence_episode_ids"))

Q: What did I eat on 2010/01/01?
GOLD: noodles
PRED: On January 1, 2010, you had lunch with a friend and then watched a movie in the evening. After that, you ate sushi
RETRIEVED IDS: ['e1_1', 'e1_3', 'e1_2']
GOLD EVIDENCE: ['e1_1']
Q: What did I eat on 2010/01/02?
GOLD: sushi
PRED: On January 2, 2010, I ate sushi with my friend at lunchtime. The episode that directly answers this question is [e2
RETRIEVED IDS: ['e2_2', 'e2_3', 'e2_1']
GOLD EVIDENCE: ['e2_1']
Q: What did I eat on 2010/01/03?
GOLD: pasta
PRED: Salad
Final answer: Salad. On January 3, 2010, I ate salad for dinner. The episode that directly answers this question is
RETRIEVED IDS: ['e3_2', 'e3_3', 'e3_1']
GOLD EVIDENCE: ['e3_1']


In [18]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 14 (delta 10), reused 14 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (14/14), 2.68 KiB | 685.00 KiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   45ef70a..99d7f26  main       -> origin/main
Updating 45ef70a..99d7f26
Fast-forward
 README.md                            |  6 ++---
 configs/prompts.yaml                 |  7 +++--
 notebooks/run_timelineqa_colab.ipynb |  5 ++--
 scripts/01_setup_env.sh              |  2 +-
 scripts/02_make_samples.py           | 14 ++++++----
 scripts/04_run_model.py              | 51 +++++++++++++++++++++++++++++++++---
 src/runners/hf_runner.py             |  9 ++++---
 7 files changed, 74 insertions(+), 20 deletions(-)


In [19]:
!python scripts/02_make_samples.py --task atomic --n 50 --seed 42

Wrote 50 atomic samples to /content/timelineqa_experiments/data/samples/atomic_n50.jsonl


In [20]:
!python scripts/04_run_model.py \
  --task atomic \
  --sample data/samples/atomic_n50.jsonl \
  --model_id Qwen/Qwen2.5-0.5B-Instruct \
  --backend hf \
  --retriever bm25 \
  --top_k 5 \
  --max_new_tokens 16 \
  --temperature 0 \
  --output outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --limit 3 \
  --debug_first_n 3

Loading one model: Qwen/Qwen2.5-0.5B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:02<00:00, 135.74it/s]
Running model:   0% 0/3 [00:00<?, ?it/s]
--- DEBUG EXAMPLE 1 ---
Question: What did I eat for lunch on 2010/01/01?
Gold: noodles
Retrieved episode ids: ['e1_1', 'e1_2', 'e1_3', 'e1_4']
Retrieved context:
[e1_1] 2010/01/01, I ate noodles for lunch.
[e1_2] 2010/01/01, I drank tea during dinner.
[e1_3] 2010/01/01, I read a book in the evening.
[e1_4] 2010/02/01, I discussed travel plans with a friend.
Prediction: noodles.
--- END DEBUG ---
Running model:  33% 1/3 [00:01<00:02,  1.35s/it]
--- DEBUG EXAMPLE 2 ---
Question: What did I eat for dinner on 2010/01/02?
Gold: noodles
Retrieved episode ids: ['e2_1', 'e2_2', 'e2_3', 'e2_4']
Retrieved context:
[e2_1] 2010/01/02, I ate noodles for dinner.
[e2_2] 2010/01/02, I drank juice during lunch.
[e2_3] 2010/01/02, I watched a movie in the evening.
[e2_4] 2010/02/02, I discussed tra

In [22]:
!cat outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 3,
  "exact_match": 1.0,
  "token_f1": 1.0,
  "denotation_accuracy": 1.0,
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 0.8313919923334652,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T18:22:13.647637+00:00"
}

In [23]:
!python scripts/04_run_model.py \
  --task atomic \
  --sample data/samples/atomic_n50.jsonl \
  --model_id Qwen/Qwen2.5-0.5B-Instruct \
  --backend hf \
  --retriever bm25 \
  --top_k 5 \
  --max_new_tokens 16 \
  --temperature 0 \
  --output outputs/predictions/atomic_n50_qwen05b_bm25.jsonl

Loading one model: Qwen/Qwen2.5-0.5B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 290/290 [00:01<00:00, 194.06it/s]
Running model: 100% 50/50 [00:28<00:00,  1.73it/s]
Saved predictions to outputs/predictions/atomic_n50_qwen05b_bm25.jsonl


In [24]:
!python scripts/05_evaluate.py \
  --predictions outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --task atomic \
  --output outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 50,
  "exact_match": 0.9,
  "token_f1": 0.9287912087912088,
  "denotation_accuracy": 0.9,
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 0.5766568181600087,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T18:24:51.083813+00:00"
}
Saved metrics to outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json


In [21]:
!python scripts/05_evaluate.py \
  --predictions outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --task atomic \
  --output outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 3,
  "exact_match": 1.0,
  "token_f1": 1.0,
  "denotation_accuracy": 1.0,
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 0.8313919923334652,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T18:22:13.647637+00:00"
}
Saved metrics to outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json


In [25]:
!python scripts/05_evaluate.py \
  --predictions outputs/predictions/atomic_n50_qwen05b_bm25.jsonl \
  --task atomic \
  --output outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 50,
  "exact_match": 0.9,
  "token_f1": 0.9287912087912088,
  "denotation_accuracy": 0.9,
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 0.5766568181600087,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T18:25:32.281459+00:00"
}
Saved metrics to outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json


In [26]:
!cat outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json

{
  "task": "atomic",
  "model_id": "Qwen/Qwen2.5-0.5B-Instruct",
  "backend": "hf",
  "retriever": "bm25",
  "top_k": 5,
  "n_examples": 50,
  "exact_match": 0.9,
  "token_f1": 0.9287912087912088,
  "denotation_accuracy": 0.9,
  "recall_at_1": 1.0,
  "recall_at_3": 1.0,
  "recall_at_5": 1.0,
  "avg_latency_sec": 0.5766568181600087,
  "predictions": "outputs/predictions/atomic_n50_qwen05b_bm25.jsonl",
  "created_at": "2026-06-30T18:25:32.281459+00:00"
}

In [27]:
!mkdir -p /content/drive/MyDrive/timelineqa_results
!cp -r outputs /content/drive/MyDrive/timelineqa_results/

In [28]:
!python scripts/06_make_results_table.py
!cat outputs/tables/baseline_results.csv

Wrote 1 rows to /content/timelineqa_experiments/outputs/tables/baseline_results.csv
task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.9,0.9287912087912088,0.9,1.0,1.0,1.0,0.5766568181600087


In [29]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/toy_qwen05b_n50
!cp -r outputs /content/drive/MyDrive/timelineqa_results/toy_qwen05b_n50/

In [30]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 11 (delta 5), reused 11 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 10.47 KiB | 2.62 MiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   99d7f26..fa45365  main       -> origin/main
Updating 99d7f26..fa45365
Fast-forward
 README.md                             |  60 +++-
 configs/experiment_plan.yaml          |  21 ++
 notebooks/run_timelineqa_colab.ipynb  |  66 +++-
 scripts/02_make_samples.py            |  19 +-
 scripts/03_prepare_timelineqa_data.py | 570 ++++++++++++++++++++++++++++++++++
 scripts/07_run_experiment_plan.py     | 289 +++++++++++++++++
 6 files changed, 999 insertions(+), 26 deletions(-)
 create mode 100644 configs/experiment_plan.yaml
 create mode 100644 scripts/03_prepare_timelineqa_data.py
 create mode 100644 scripts/07_run_experiment_plan.py


In [31]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --limit 3 \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_185648.log
Dry run: False
Resume: True
Limit forwarded to model runs: 3

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 50 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/ti

In [32]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_190053.log
Dry run: False
Resume: True

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 50 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/Tim

In [33]:
%cd /content/timelineqa_experiments

!rm outputs/metrics/real_atomic_n50_*_bm25_metrics.json
!rm outputs/predictions/real_atomic_n50_*_bm25.jsonl

/content


In [34]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_190334.log
Dry run: False
Resume: True

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 50 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/Tim

In [35]:
!cat outputs/tables/baseline_results.csv

task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.9,0.9287912087912088,0.9,1.0,1.0,1.0,0.5766568181600087
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.14,0.3387344877344877,0.18,0.88,1.0,1.0,0.5695098818799306
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,50,0.46,0.646943722943723,0.56,0.88,1.0,1.0,0.40484363841991583
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,50,0.0,0.0,0.0,0.88,1.0,1.0,0.07589687850004338
atomic,TinyLlama/TinyLlama-1.1B-Chat-v1.0,hf,bm25,5,50,0.08,0.26713963813963815,0.24,0.88,1.0,1.0,0.5488053821400172


In [36]:
%cd /content/timelineqa_experiments
!rm -f outputs/metrics/atomic_n50_qwen05b_bm25_metrics.json
!python scripts/06_make_results_table.py
!cat outputs/tables/baseline_results.csv

/content
Wrote 4 rows to /content/timelineqa_experiments/outputs/tables/baseline_results.csv
task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.14,0.3387344877344877,0.18,0.88,1.0,1.0,0.5695098818799306
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,50,0.46,0.646943722943723,0.56,0.88,1.0,1.0,0.40484363841991583
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,50,0.0,0.0,0.0,0.88,1.0,1.0,0.07589687850004338
atomic,TinyLlama/TinyLlama-1.1B-Chat-v1.0,hf,bm25,5,50,0.08,0.26713963813963815,0.24,0.88,1.0,1.0,0.5488053821400172


In [37]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/final_real_atomic_n50
!cp -r outputs /content/drive/MyDrive/timelineqa_results/final_real_atomic_n50/
!cp -r data/samples /content/drive/MyDrive/timelineqa_results/final_real_atomic_n50/

In [38]:
import json

for fname in [
    "outputs/predictions/real_atomic_n50_qwen15b_bm25.jsonl",
    "outputs/predictions/real_atomic_n50_smollm17b_bm25.jsonl",
]:
    print("\nFILE:", fname)
    with open(fname, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            row = json.loads(line)
            print("="*80)
            print("Q:", row["question"])
            print("GOLD:", row["gold_answer"])
            print("PRED:", row["predicted_answer"])


FILE: outputs/predictions/real_atomic_n50_qwen15b_bm25.jsonl
Q: What exercise I did on 2012/05/24?
GOLD: hiking
PRED: Hiking.
Q: Did I talk to anyone on 1987/11/22?
GOLD: yes
PRED: No. Food item: Date
Q: how long did I talk to Nevaeh, Allison in the early evening on 1983/04/04?
GOLD: 45 minutes
PRED: 45 minutes.
Q: How long did I stay in Los Angeles, US?
GOLD: 3
PRED: 3 days.
Q: Did I watch tv on 1996/03/06?
GOLD: yes
PRED: Yes. Food item: TV

FILE: outputs/predictions/real_atomic_n50_smollm17b_bm25.jsonl
Q: What exercise I did on 2012/05/24?
GOLD: hiking
PRED: 
Q: Did I talk to anyone on 1987/11/22?
GOLD: yes
PRED: 
Q: how long did I talk to Nevaeh, Allison in the early evening on 1983/04/04?
GOLD: 45 minutes
PRED: 
Q: How long did I stay in Los Angeles, US?
GOLD: 3
PRED: 
Q: Did I watch tv on 1996/03/06?
GOLD: yes
PRED: 


In [39]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 12 (delta 9), reused 12 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.48 KiB | 282.00 KiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   fa45365..4b4f47a  main       -> origin/main
Updating fa45365..4b4f47a
Fast-forward
 README.md                    | 17 +++++++++++++++++
 configs/prompts.yaml         |  1 -
 scripts/04_run_model.py      | 36 +++++++++++++++++++++++++++++++-----
 src/runners/base_runner.py   | 13 +++++++++++--
 src/runners/hf_runner.py     | 44 +++++++++++++++++++++++++++++++++++---------
 src/runners/ollama_runner.py | 10 +++++++---
 6 files changed, 101 insertions(+), 20 deletions(-)


In [40]:
!rm -f outputs/metrics/real_atomic_n50_*_bm25_metrics.json
!rm -f outputs/predictions/real_atomic_n50_*_bm25.jsonl
!rm -f outputs/tables/baseline_results.csv

In [41]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --limit 3 \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_191752.log
Dry run: False
Resume: True
Limit forwarded to model runs: 3

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 50 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/ti

In [42]:
import json

path = "outputs/predictions/real_atomic_n50_smollm17b_bm25.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        row = json.loads(line)
        print("="*80)
        print("Q:", row["question"])
        print("GOLD:", row["gold_answer"])
        print("RAW:", row.get("raw_predicted_answer"))
        print("CLEAN:", row["predicted_answer"])

Q: What exercise I did on 2012/05/24?
GOLD: hiking
RAW: I did hiking on 2012/05/24.
CLEAN: I did hiking on 2012/05/24.
Q: Did I talk to anyone on 1987/11/22?
GOLD: yes
RAW: No
CLEAN: no
Q: how long did I talk to Nevaeh, Allison in the early evening on 1983/04/04?
GOLD: 45 minutes
RAW: 45 minutes
CLEAN: 45 minutes


In [43]:
!rm -f outputs/metrics/real_atomic_n50_*_bm25_metrics.json
!rm -f outputs/predictions/real_atomic_n50_*_bm25.jsonl
!rm -f outputs/tables/baseline_results.csv

In [44]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_192027.log
Dry run: False
Resume: True

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 50 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/Tim

In [45]:
!cat outputs/tables/baseline_results.csv

task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.22,0.41036021872863976,0.24,0.88,1.0,1.0,0.32054984447995594
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,50,0.42,0.6047330447330448,0.52,0.88,1.0,1.0,0.2973623302800479
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,50,0.3,0.48217543859649126,0.38,0.88,1.0,1.0,0.25262842816004194
atomic,TinyLlama/TinyLlama-1.1B-Chat-v1.0,hf,bm25,5,50,0.24,0.3668221484397955,0.26,0.88,1.0,1.0,0.5224364993199196


In [46]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/metrics /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/predictions /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp data/samples/real_atomic_n50.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/

In [47]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/metrics /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/predictions /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp data/samples/real_atomic_n50.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/

In [48]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/metrics /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp -r outputs/predictions /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/
!cp data/samples/real_atomic_n50.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n50/

In [49]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 370 bytes | 185.00 KiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   4b4f47a..37e6d6a  main       -> origin/main
Updating 4b4f47a..37e6d6a
Fast-forward
 configs/experiment_plan.yaml | 7 +------
 1 file changed, 1 insertion(+), 6 deletions(-)


In [50]:
!rm -f outputs/metrics/real_atomic_n100_*_bm25_metrics.json
!rm -f outputs/predictions/real_atomic_n100_*_bm25.jsonl

!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

!cat outputs/tables/baseline_results.csv

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_192916.log
Dry run: False
Resume: True

[1/4] Preparing data for n=100...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 100 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n100.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/

In [53]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n100_top2
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n100_top2/
!cp outputs/metrics/real_atomic_n100_*_bm25_metrics.json /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n100_top2/
!cp outputs/predictions/real_atomic_n100_*_bm25.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n100_top2/
!cp data/samples/real_atomic_n100.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n100_top2/

In [54]:
!rm -f outputs/metrics/real_atomic_n500_*_bm25_metrics.json
!rm -f outputs/predictions/real_atomic_n500_*_bm25.jsonl

In [55]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/experiment_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results
!cat outputs/tables/baseline_results.csv

Experiment plan: configs/experiment_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_193523.log
Dry run: False
Resume: True

[1/4] Preparing data for n=500...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task atomic --n 500 --seed 42 --max_episodes_per_question 100 --output data/samples/real_atomic_n500.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/

In [57]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp outputs/metrics/real_atomic_n500_qwen15b_bm25_metrics.json /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp outputs/predictions/real_atomic_n500_qwen15b_bm25.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp data/samples/real_atomic_n500.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/

In [58]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp outputs/metrics/real_atomic_n500_qwen15b_bm25_metrics.json /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp outputs/predictions/real_atomic_n500_qwen15b_bm25.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/
!cp data/samples/real_atomic_n500.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_atomic_n500_qwen15b/

In [59]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 11 (delta 5), reused 7 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 43.70 KiB | 2.91 MiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   633d941..af3bb81  main       -> origin/main
Updating 633d941..af3bb81
Fast-forward
 README.md                             |   24 +
 configs/multihop_baseline_plan.yaml   |   14 +
 notebooks/run_timelineqa_colab.ipynb  | 4270 ++++++++++++++++++++++++++++++++-
 scripts/03_prepare_timelineqa_data.py |  465 +++-
 4 files changed, 4663 insertions(+), 110 deletions(-)
 create mode 100644 configs/multihop_baseline_plan.yaml


In [60]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/multihop_baseline_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/multihop_baseline_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_201655.log
Dry run: False
Resume: True

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task multihop --n 50 --seed 42 --max_episodes_per_question 200 --output data/samples/real_multihop_n50.jsonl
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/o

In [61]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 9 (delta 8), reused 9 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 1.97 KiB | 503.00 KiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   af3bb81..9a46a99  main       -> origin/main
Updating af3bb81..9a46a99
Fast-forward
 README.md                               |  10 +++
 configs/multihop_baseline_n25_plan.yaml |  15 +++++
 configs/multihop_baseline_plan.yaml     |   1 +
 scripts/03_prepare_timelineqa_data.py   | 115 +++++++++++++++++++++++++-------
 scripts/07_run_experiment_plan.py       |   5 +-
 5 files changed, 120 insertions(+), 26 deletions(-)
 create mode 100644 configs/multihop_baseline_n25_plan.yaml


In [62]:
!python scripts/03_prepare_timelineqa_data.py \
  --task multihop \
  --n 5 \
  --seed 42 \
  --max_episodes_per_question 200 \
  --max_seed_attempts 3 \
  --output data/samples/real_multihop_n5.jsonl

Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/travel-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/travel_dining-qa.csv
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/travel_places_visited-qa.csv
Using existing generated TimelineQA lifelog: /content/timelineqa_expe

In [63]:
!rm -f outputs/metrics/real_multihop_n50_*_bm25_metrics.json
!rm -f outputs/predictions/real_multihop_n50_*_bm25.jsonl

!python scripts/07_run_experiment_plan.py \
  --plan configs/multihop_baseline_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Experiment plan: configs/multihop_baseline_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_202242.log
Dry run: False
Resume: True

[1/4] Preparing data for n=50...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task multihop --n 50 --seed 42 --max_episodes_per_question 200 --output data/samples/real_multihop_n50.jsonl --max_seed_attempts 20
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed_csvs: /content/t

In [64]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/
!cp outputs/metrics/real_multihop_n50_qwen15b_bm25_metrics.json /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/
!cp outputs/predictions/real_multihop_n50_qwen15b_bm25.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/
!cp data/samples/real_multihop_n50.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/

In [65]:
!python scripts/07_run_experiment_plan.py \
  --plan configs/multihop_baseline_n100_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

Traceback (most recent call last):
  File "/content/timelineqa_experiments/scripts/07_run_experiment_plan.py", line 292, in <module>
    main()
  File "/content/timelineqa_experiments/scripts/07_run_experiment_plan.py", line 235, in main
    plan = read_yaml(plan_path)
           ^^^^^^^^^^^^^^^^^^^^
  File "/content/timelineqa_experiments/src/utils/io.py", line 39, in read_yaml
    with Path(path).open("r", encoding="utf-8") as handle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1013, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/timelineqa_experiments/configs/multihop_baseline_n100_plan.yaml'


In [66]:
from pathlib import Path

Path("/content/timelineqa_experiments/configs/multihop_baseline_n100_plan.yaml").write_text("""task: multihop
data_source: real_timelineqa
seed: 42
sample_sizes: [100]
max_episodes_per_question: 200
max_seed_attempts: 20
retriever: bm25
top_k: 10
max_new_tokens: 32
temperature: 0
backend: hf

models:
  - model_id: Qwen/Qwen2.5-1.5B-Instruct
    short_name: qwen15b
""")

print("Created n100 multihop plan")

Created n100 multihop plan


In [67]:
!cat /content/timelineqa_experiments/configs/multihop_baseline_n100_plan.yaml

task: multihop
data_source: real_timelineqa
seed: 42
sample_sizes: [100]
max_episodes_per_question: 200
max_seed_attempts: 20
retriever: bm25
top_k: 10
max_new_tokens: 32
temperature: 0
backend: hf

models:
  - model_id: Qwen/Qwen2.5-1.5B-Instruct
    short_name: qwen15b


In [68]:
%cd /content/timelineqa_experiments

!python scripts/07_run_experiment_plan.py \
  --plan configs/multihop_baseline_n100_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

/content
Experiment plan: configs/multihop_baseline_n100_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_202945.log
Dry run: False
Resume: True

[1/4] Preparing data for n=100...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task multihop --n 100 --seed 42 --max_episodes_per_question 200 --output data/samples/real_multihop_n100.jsonl --max_seed_attempts 20
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  multihop_seed

In [69]:
!cat outputs/tables/baseline_results.csv

task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,100,0.48,0.610234943419154,0.55,0.9,0.99,1.0,0.2850821696699677
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,100,0.38,0.5094631047315258,0.43,0.9,0.99,1.0,0.23416169955999067
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,500,0.45,0.6111613520981942,0.5,0.904,0.992,1.0,0.2889145701739981
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.22,0.41036021872863976,0.24,0.88,1.0,1.0,0.32054984447995594
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,50,0.42,0.6047330447330448,0.52,0.88,1.0,1.0,0.2973623302800479
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,50,0.3,0.48217543859649126,0.38,0.88,1.0,1.0,0.25262842816004194
atomic,TinyLlama/TinyLlama-1.1B-Chat-v1.0,hf,bm25,5,50,0.24,0.3668221484397955,0.26,0.88,1.0,1.0,0.5224364993199196
multihop,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,10,100,0.15,0.1946666666

In [70]:
%%writefile configs/multihop_baseline_n100_allmodels_plan.yaml
task: multihop
data_source: real_timelineqa
seed: 42
sample_sizes: [100]
max_episodes_per_question: 200
max_seed_attempts: 20
retriever: bm25
top_k: 10
max_new_tokens: 32
temperature: 0
backend: hf

models:
  - model_id: Qwen/Qwen2.5-0.5B-Instruct
    short_name: qwen05b
  - model_id: Qwen/Qwen2.5-1.5B-Instruct
    short_name: qwen15b
  - model_id: HuggingFaceTB/SmolLM2-1.7B-Instruct
    short_name: smollm17b
  - model_id: TinyLlama/TinyLlama-1.1B-Chat-v1.0
    short_name: tinyllama11b

Writing configs/multihop_baseline_n100_allmodels_plan.yaml


In [71]:
%cd /content/timelineqa_experiments

!python scripts/07_run_experiment_plan.py \
  --plan configs/multihop_baseline_n100_allmodels_plan.yaml \
  --resume \
  --copy_to_drive /content/drive/MyDrive/timelineqa_results

/content
Experiment plan: configs/multihop_baseline_n100_allmodels_plan.yaml
Log file: outputs/logs/experiment_plan_20260630_203914.log
Dry run: False
Resume: True

[1/4] Preparing data for n=100...
$ /usr/bin/python3 scripts/03_prepare_timelineqa_data.py --task multihop --n 100 --seed 42 --max_episodes_per_question 200 --output data/samples/real_multihop_n100.jsonl --max_seed_attempts 20
Official TimelineQA files discovered:
  generator: /content/timelineqa_experiments/original/TimelineQA/src/generateDB.py
  atomic_converter: /content/timelineqa_experiments/original/TimelineQA/src/create_qa_data.py
  multihop_generator: /content/timelineqa_experiments/original/TimelineQA/multihopQA/multihopQA.py
  multihop_queryfile: /content/timelineqa_experiments/original/TimelineQA/multihopQA/queryfile.csv
  templates: /content/timelineqa_experiments/original/TimelineQA/data/templates.json
  multihop_seed_csvs: /content/timelineqa_experiments/original/TimelineQA/data/multihop/marriages-qa.csv
  mul

In [72]:
!cat outputs/tables/baseline_results.csv

task,model_id,backend,retriever,top_k,n_examples,exact_match,token_f1,denotation_accuracy,recall_at_1,recall_at_3,recall_at_5,avg_latency_sec
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,100,0.48,0.610234943419154,0.55,0.9,0.99,1.0,0.2850821696699677
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,100,0.38,0.5094631047315258,0.43,0.9,0.99,1.0,0.23416169955999067
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,500,0.45,0.6111613520981942,0.5,0.904,0.992,1.0,0.2889145701739981
atomic,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,5,50,0.22,0.41036021872863976,0.24,0.88,1.0,1.0,0.32054984447995594
atomic,Qwen/Qwen2.5-1.5B-Instruct,hf,bm25,5,50,0.42,0.6047330447330448,0.52,0.88,1.0,1.0,0.2973623302800479
atomic,HuggingFaceTB/SmolLM2-1.7B-Instruct,hf,bm25,5,50,0.3,0.48217543859649126,0.38,0.88,1.0,1.0,0.25262842816004194
atomic,TinyLlama/TinyLlama-1.1B-Chat-v1.0,hf,bm25,5,50,0.24,0.3668221484397955,0.26,0.88,1.0,1.0,0.5224364993199196
multihop,Qwen/Qwen2.5-0.5B-Instruct,hf,bm25,10,100,0.07,0.0875982683

In [73]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels
!cp outputs/tables/baseline_results.csv /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/
!cp outputs/metrics/real_multihop_n100_*_bm25_metrics.json /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/
!cp outputs/predictions/real_multihop_n100_*_bm25.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/
!cp data/samples/real_multihop_n100.jsonl /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

In [74]:
%cd /content/timelineqa_experiments

/content


In [75]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b

!cp outputs/tables/baseline_results.csv \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/

!cp outputs/metrics/real_multihop_n50_qwen15b_bm25_metrics.json \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/

!cp outputs/predictions/real_multihop_n50_qwen15b_bm25.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/

!cp data/samples/real_multihop_n50.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b/

In [76]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b

!cp outputs/tables/baseline_results.csv \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b/

!cp outputs/metrics/real_multihop_n100_qwen15b_bm25_metrics.json \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b/

!cp outputs/predictions/real_multihop_n100_qwen15b_bm25.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b/

!cp data/samples/real_multihop_n100.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b/

In [77]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot

!cp outputs/tables/baseline_results.csv \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot/

!cp -r outputs/metrics \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot/

!cp -r outputs/predictions \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot/

!cp -r data/samples \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot/

In [78]:
!ls /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n50_qwen15b
!ls /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_qwen15b
!ls /content/drive/MyDrive/timelineqa_results/baseline_v1_combined_atomic_multihop_snapshot

baseline_results.csv	 real_multihop_n50_qwen15b_bm25.jsonl
real_multihop_n50.jsonl  real_multihop_n50_qwen15b_bm25_metrics.json
baseline_results.csv	  real_multihop_n100_qwen15b_bm25.jsonl
real_multihop_n100.jsonl  real_multihop_n100_qwen15b_bm25_metrics.json
baseline_results.csv  metrics  predictions  samples


In [52]:
%cd /content/timelineqa_experiments
!git pull

/content
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 363 bytes | 363.00 KiB/s, done.
From https://github.com/KYPKRISHNAREDDY/timelineqa_experiments
   37e6d6a..633d941  main       -> origin/main
Updating 37e6d6a..633d941
Fast-forward
 configs/experiment_plan.yaml | 4 +---
 1 file changed, 1 insertion(+), 3 deletions(-)


In [79]:
%cd /content/timelineqa_experiments

!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels

!cp outputs/tables/baseline_results.csv \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

!cp outputs/metrics/real_multihop_n100_*_bm25_metrics.json \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

!cp outputs/predictions/real_multihop_n100_*_bm25.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

!cp data/samples/real_multihop_n100.jsonl \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

!cp configs/multihop_baseline_n100_allmodels_plan.yaml \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_real_multihop_n100_allmodels/

/content


In [80]:
!mkdir -p /content/drive/MyDrive/timelineqa_results/baseline_v1_FINAL_atomic_multihop

!cp outputs/tables/baseline_results.csv \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_FINAL_atomic_multihop/

!cp -r outputs/metrics \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_FINAL_atomic_multihop/

!cp -r outputs/predictions \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_FINAL_atomic_multihop/

!cp -r data/samples \
  /content/drive/MyDrive/timelineqa_results/baseline_v1_FINAL_atomic_multihop/